# 1. Task 3 Demo — Arabic Query to English Idiom

This notebook demonstrates the final Task 3 retrieval model.

Input:
- Arabic contextual sentence

Output:
- top retrieved English idioms with similarity scores

This notebook is inference-only and uses saved artifacts from the benchmark notebook.

In [9]:
# 1.1 Setup Paths

from pathlib import Path

ARTIFACTS_DIR = Path("../artifacts/task3")
DATA_DIR = ARTIFACTS_DIR / "data"
MODELS_DIR = ARTIFACTS_DIR / "models"
DEMO_DIR = ARTIFACTS_DIR / "demo"

DEMO_DIR.mkdir(parents=True, exist_ok=True)

BANK_PATH = DATA_DIR / "task3_bank.csv"
MODEL_PATH = MODELS_DIR / "task3_e5_finetuned_v1"

print("Bank exists :", BANK_PATH.exists())
print("Model exists:", MODEL_PATH.exists())
print("Demo dir    :", DEMO_DIR.resolve())

Bank exists : True
Model exists: True
Demo dir    : C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\artifacts\task3\demo


# 2. Load Demo Artifacts

Load the saved semantic bank and the fine-tuned retrieval model.

In [10]:
# 2.1 Load Bank and Model

import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

bank_df = pd.read_csv(BANK_PATH, encoding="utf-8-sig")
model = SentenceTransformer(str(MODEL_PATH), device=device)

print("Device    :", device)
print("Bank shape :", bank_df.shape)
print("Columns    :", bank_df.columns.tolist())

Device    : cuda
Bank shape : (69549, 3)
Columns    : ['bank_text_ar', 'source_type', 'target_en']


# 3. Prepare Bank Text (E5 Formatting)

E5 models require specific prefixes:
- "query:" for input queries
- "passage:" for candidate texts (bank)

Here we prepare the bank texts accordingly.

In [11]:
# 3.1 Prepare Bank Text

# Convert Arabic bank text into E5 "passage" format
# This ensures consistency with training and improves retrieval quality
bank_df["bank_text_e5"] = "passage: " + bank_df["bank_text_ar"]

# Quick check
bank_df[["bank_text_ar", "bank_text_e5"]].head()

,bank_text_ar,bank_text_e5
0,يؤسفني القول، تم إلغاء الاجتماع.,passage: يؤسفني القول، تم إلغاء الاجتماع.
1,تأكيد إلغاء الاجتماع مع بعض الأسى.,passage: تأكيد إلغاء الاجتماع مع بعض الأسى.
2,طريقة غير رسمية للتأكيد على أن شيئًا ما صحيح، ...,passage: طريقة غير رسمية للتأكيد على أن شيئًا ...
3,بالتأكيد، أعذارك صادقة جدًا - لم يقلها أحد أبدًا.,passage: بالتأكيد، أعذارك صادقة جدًا - لم يقله...
4,استخدام ساخر يعني أن الأعذار غير حقيقية، مع ال...,passage: استخدام ساخر يعني أن الأعذار غير حقيق...


# 4. Encode Bank Embeddings

We encode all candidate idioms once and store embeddings in memory.

This avoids recomputing embeddings for every query and makes the demo fast.

In [12]:
# 4.1 Encode Bank Embeddings

from tqdm import tqdm

# Convert to list for model input
bank_texts = bank_df["bank_text_e5"].tolist()

print("Encoding bank... this runs once only")

# Encode all bank entries into dense vectors
# batch_size can be increased if GPU memory allows
bank_embeddings = model.encode(
    bank_texts,
    batch_size=64,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Bank embeddings shape:", bank_embeddings.shape)

Encoding bank... this runs once only


Batches:   0%|          | 0/1087 [00:00<?, ?it/s]

Bank embeddings shape: torch.Size([69549, 768])


# 5. Retrieval Function

Create a simple function that:
- formats the Arabic query
- encodes it
- compares it with the bank embeddings
- returns the top retrieved idioms

In [13]:
# 5.1 Build Demo Retrieval Function

import pandas as pd
from sentence_transformers import util

def retrieve_idiom_ar(query_text: str, top_k: int = 5) -> pd.DataFrame:
    """
    Retrieve the top-k English idioms for an Arabic query.

    Steps:
    1. format the query using the E5 "query:" prefix
    2. encode the query into the same embedding space as the bank
    3. compute cosine similarity against all bank embeddings
    4. return unique idioms with their best-ranked supporting bank text
    """

    # Format query for E5 retrieval
    formatted_query = "query: " + str(query_text).strip()

    # Encode the query into a dense embedding
    query_embedding = model.encode(
        [formatted_query],
        convert_to_tensor=True,
        show_progress_bar=False
    )

    # Compute cosine similarity between the query and all bank entries
    scores = util.cos_sim(query_embedding, bank_embeddings)[0]

    # Retrieve more than top_k initially, then deduplicate by idiom
    # This helps avoid repeated idioms from multiple bank rows
    top_indices = scores.topk(k=min(top_k * 5, len(bank_df))).indices.tolist()

    rows = []
    seen_idioms = set()

    for idx in top_indices:
        idiom = bank_df.iloc[idx]["target_en"]

        # Keep only the first occurrence of each idiom
        if idiom in seen_idioms:
            continue

        seen_idioms.add(idiom)

        rows.append({
            "rank": len(rows) + 1,
            "idiom": idiom,
            "source_type": bank_df.iloc[idx]["source_type"],
            "bank_text_ar": bank_df.iloc[idx]["bank_text_ar"],
            "score": float(scores[idx])
        })

        # Stop once we have the requested number of unique idioms
        if len(rows) >= top_k:
            break

    return pd.DataFrame(rows)

---

# 5.2 One Demo Query

Run one Arabic query to verify that the demo pipeline works correctly.

In [14]:
# 5.2 Try One Demo Query

demo_query = "هو قرر يواجه المشكلة مباشرة"

demo_results = retrieve_idiom_ar(demo_query, top_k=5)

print("Query:", demo_query)
print(demo_results.to_string(index=False))

Query: هو قرر يواجه المشكلة مباشرة
 rank               idiom       source_type                                                          bank_text_ar    score
    1     bite the bullet   example_meaning                         اختارت مواجهة قرار محفوف بالمخاطر لكنه ضروري. 0.571675
    2        skate around canonical_meaning                            تجنب التعامل مباشرة مع مشكلة أو قضية صعبة. 0.564446
    3             take on           example       وافق الرئيس التنفيذي على مواجهة تحدي التوسع في الأسواق الجديدة. 0.550896
    4 rip off the bandage canonical_meaning مواجهة والتعامل مع موقف غير سار أو صعب بسرعة وبتصميم، بدلاً من تجنبه. 0.539900
    5     take the plunge canonical_meaning       اتخاذ قرار حاسم بالبدء في شيء مهم أو صعب بعد التردد أو التفكير. 0.535641


# 6. Multi-Query Demo

Test the demo on a small set of Arabic queries.

This helps verify:
- general behavior
- retrieval quality
- result readability

In [15]:
# 6.1 Run Multiple Demo Queries

# Example Arabic test queries for demonstration
demo_queries = [
    "هو حاول يهدئ الوضع بينهم",
    "الموضوع خرج عن السيطرة تماما",
    "هو قرر يواجه المشكلة مباشرة",
    "كان واضح انه يخفي شيء"
]

# Run retrieval for each query and print results
for query in demo_queries:
    print("\n" + "=" * 80)
    print("Query:", query)

    results = retrieve_idiom_ar(query, top_k=3)
    print(results[["rank", "idiom", "score"]].to_string(index=False))


Query: هو حاول يهدئ الوضع بينهم
 rank           idiom    score
    1 steady the ship 0.694418
    2       hose down 0.672186
    3      smooth out 0.637330

Query: الموضوع خرج عن السيطرة تماما
 rank                  idiom    score
    1 Frankenstein's monster 0.715209
    2          lose the plot 0.598746
    3      get the better of 0.584704

Query: هو قرر يواجه المشكلة مباشرة
 rank           idiom    score
    1 bite the bullet 0.571675
    2    skate around 0.564446
    3         take on 0.550896

Query: كان واضح انه يخفي شيء
 rank           idiom    score
    1 on the down-low 0.725933
    2     drop a hint 0.716707
    3   come to light 0.691937


# 6.2 Save One Demo Example

Save one demo query result as a small CSV example for reuse.

In [16]:
# 6.2 Save One Demo Example

demo_query = "هو قرر يواجه المشكلة مباشرة"
demo_results = retrieve_idiom_ar(demo_query, top_k=5)

demo_example_path = DEMO_DIR / "demo_example.csv"
demo_results.to_csv(demo_example_path, index=False, encoding="utf-8-sig")

print("Saved demo example:", demo_example_path)

Saved demo example: ..\artifacts\task3\demo\demo_example.csv


In [17]:
from pathlib import Path
import numpy as np
import torch

ARTIFACTS_DIR = Path("../artifacts/task3")
DATA_DIR = ARTIFACTS_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Convert tensor to numpy if needed
if torch.is_tensor(bank_embeddings):
    bank_embeddings_np = bank_embeddings.cpu().numpy()
else:
    bank_embeddings_np = np.asarray(bank_embeddings)

np.save(DATA_DIR / "task3_bank_embeddings.npy", bank_embeddings_np)

print("Saved:", DATA_DIR / "task3_bank_embeddings.npy")
print("Shape :", bank_embeddings_np.shape)

Saved: ..\artifacts\task3\data\task3_bank_embeddings.npy
Shape : (69549, 768)
